![Health-Informatics: A Python Tutorial](../../Image/chapter-banner.png)


# Module 7: Common Data Models (OMOP CDM)



**Health Informatics in Python** · Part II: Interoperability & Data Engineering · Module 7 of 16

---



Every health system stores data differently. A **Common Data Model (CDM)** solves
this by defining one standardized schema and vocabulary, so the *same analysis
code* runs unchanged across many institutions. The **OMOP CDM** (from the OHDSI
community) is the most widely used for observational research. This module maps our
synthetic EHR into OMOP.


## Learning objectives

By the end of this module you will be able to:

1. Explain **why** a common data model matters for multi-site research.
2. Describe the core **OMOP tables**: `person`, `visit_occurrence`,
   `condition_occurrence`, `drug_exposure`, `measurement`, and the `concept` vocabulary.
3. **Map source data to OMOP** using vocabulary tables and standard concept IDs.
4. Run a **standardized analytic query** against the CDM.
5. Name alternative CDMs (**i2b2, PCORnet, Sentinel**) and when they're used.


## Dataset

We reuse the synthetic EHR and transform it into a **miniature OMOP CDM**. The
`concept` IDs shown are **illustrative** standard concepts (the real OMOP vocabulary
lives in OHDSI Athena); a couple — gender and common conditions — match real
standard concept IDs, but treat the table as a teaching stand-in, not the source of truth.


In [16]:
# --- Self-contained synthetic EHR generator (identical to Part I) ---
# This is the SAME generator from Module 1. We rebuild the fake hospital database
# here so this notebook can run on its own. Nothing here is a real patient.
import numpy as np
import pandas as pd

# This function generates a synthetic electronic health record (EHR) dataset.
# It creates mock tables for patients, encounters, observations, conditions, and medications,
# all with plausible structure, but not referencing any real patients or requiring external data.
def make_synthetic_ehr(n_patients=200, seed=42):
    """
    Generate a small, internally-consistent synthetic EHR dataset.
    Returns a dict of linked DataFrames: patients, encounters, observations,
    conditions, and medications. This is for illustrative, educational use.
    """
    rng = np.random.default_rng(seed)  # same seed => same fake people every run

    # --- patients table ---
    # Create a DataFrame with n_patients patients, each with a name, sex, age, and birth year.
    first = ["Ava","Liam","Noah","Mia","Zoe","Omar","Ivan","Sara","Leo","Nina",
             "Ruth","Kai","Yara","Theo","Ida","Sam","Ana","Eli","Rex","Uma"]
    last  = ["Khan","Ortiz","Chen","Diaz","Patel","Ali","Brown","Nash","Reed","Vega",
             "Cole","Frost","Grant","Hale","Iqbal","Jain","Kerr","Lund","Mora","Park"]
    sexes = rng.choice(["male", "female"], size=n_patients, p=[0.49, 0.51])
    ages  = rng.integers(18, 90, size=n_patients)
    # Columns:
    #   patient_id  — unique ID, e.g. "P1000", "P1001"
    #   given_name / family_name — randomly chosen from the name lists
    #   sex, age — pre-generated arrays
    #   birth_year — derived from age, assuming the reference year 2026
    patients = pd.DataFrame({
        "patient_id": [f"P{1000+i}" for i in range(n_patients)],
        "given_name": rng.choice(first, size=n_patients),
        "family_name": rng.choice(last, size=n_patients),
        "sex": sexes,
        "age": ages,
        "birth_year": 2026 - ages,
    })

    # --- encounters table ---
    # Each patient gets 1–4 visits. Types and dates are random but internally consistent.
    #   encounter_id    — unique, zero-padded (E00001, E00002, ...)
    #   encounter_type  — ambulatory 55%, emergency 15%, inpatient 10%, wellness 20%
    #   date            — a random day in a 3-year window starting 2023-01-01
    enc_rows = []
    enc_types = ["ambulatory", "emergency", "inpatient", "wellness"]
    for pid in patients["patient_id"]:
        for _ in range(rng.integers(1, 5)):  # 1 to 4 encounters per patient
            day = rng.integers(0, 365*3)     # offset within 3 years
            enc_rows.append({
                "encounter_id": f"E{len(enc_rows)+1:05d}",
                "patient_id": pid,
                "encounter_type": rng.choice(enc_types, p=[0.55, 0.15, 0.10, 0.20]),
                "date": (pd.Timestamp("2023-01-01") + pd.Timedelta(days=int(day))).date(),
            })
    encounters = pd.DataFrame(enc_rows)

    # --- observations table ---
    # Labs and vitals attached to encounters. Each tuple is (name, unit, lo, hi).
    # A value is drawn uniformly in [lo, hi]; 70% of (encounter, measure) pairs exist
    # so the table looks sparse the way a real EHR does.
    obs_defs = [
        ("Body height", "cm", 150, 195),
        ("Body weight", "kg", 50, 110),
        ("Systolic blood pressure", "mmHg", 100, 165),
        ("Heart rate", "/min", 55, 100),
        ("Hemoglobin A1c", "%", 4.8, 9.5),
    ]
    obs_rows = []
    for _, e in encounters.iterrows():
        for name, unit, lo, hi in obs_defs:
            if rng.random() < 0.7:
                obs_rows.append({
                    "observation_id": f"O{len(obs_rows)+1:06d}",
                    "encounter_id": e["encounter_id"],
                    "patient_id": e["patient_id"],
                    "observation": name,
                    "value": round(float(rng.uniform(lo, hi)), 1),
                    "unit": unit,
                    "date": e["date"],
                })
    observations = pd.DataFrame(obs_rows)

    # --- conditions table ---
    # Each patient is assigned 0–3 unique diagnoses from a small pool.
    cond_pool = ["Essential hypertension", "Type 2 diabetes mellitus", "Asthma",
                 "Acute bronchitis", "Major depressive disorder", "Osteoarthritis",
                 "Chronic kidney disease", "Anemia"]
    cond_rows = []
    for pid in patients["patient_id"]:
        for c in rng.choice(cond_pool, size=rng.integers(0, 4), replace=False):
            cond_rows.append({
                "condition_id": f"C{len(cond_rows)+1:05d}",
                "patient_id": pid,
                "condition": c,
            })
    conditions = pd.DataFrame(cond_rows)

    # --- medications table ---
    # Same pattern as conditions: 0–3 unique drugs per patient.
    med_pool = ["Lisinopril", "Metformin", "Albuterol", "Atorvastatin",
                "Sertraline", "Amoxicillin", "Ibuprofen", "Hydrochlorothiazide"]
    med_rows = []
    for pid in patients["patient_id"]:
        for m in rng.choice(med_pool, size=rng.integers(0, 4), replace=False):
            med_rows.append({
                "medication_id": f"M{len(med_rows)+1:05d}",
                "patient_id": pid,
                "medication": m,
            })
    medications = pd.DataFrame(med_rows)

    return {
        "patients": patients,
        "encounters": encounters,
        "observations": observations,
        "conditions": conditions,
        "medications": medications,
    }

# Generate the five linked tables. Each notebook in this series starts from here.
ehr = make_synthetic_ehr()
print("Tables:", ", ".join(f"{k} ({len(v)} rows)" for k, v in ehr.items()))


Tables: patients (200 rows), encounters (511 rows), observations (1804 rows), conditions (304 rows), medications (276 rows)


## 7.1 Why a common data model

A Common Data Model (CDM) standardizes database structure across multiple sites.
Without a CDM, each institution in a multi-site study needs to write its own custom data extraction code, creating duplication and inconsistency.
With a CDM, every site converts its local data into the same format just one time.
After this initial mapping, the same analytical tools and scripts can be used everywhere, greatly simplifying multicenter research.
This is the approach used by OHDSI network studies: map once, then share analyses.

**The initial investment in mapping enables efficient, reusable, and shareable analytics across sites.**


## 7.2 The OMOP vocabulary (`concept`)

At the heart of the OMOP Common Data Model (CDM) lies a **standardized vocabulary**, which plays a crucial role in harmonizing clinical data from diverse sources. In OMOP, every meaningful clinical entity—whether it pertains to a diagnosis, measurement, medication, or demographic attribute—is represented by a numeric identifier called a `concept_id`.

This approach ensures that every clinical fact stored in an OMOP database is linked to a specific, well-defined concept. Instead of storing local or proprietary codes (such as ICD-10 for diagnoses, SNOMED for conditions, LOINC for lab tests, or RxNorm for medications), OMOP requires that all source codes be mapped to **standard concepts** from its extensive vocabulary. Each standard concept is assigned a unique `concept_id` and grouped into meaningful domains (e.g., Condition, Measurement, Drug, Gender).

By forcing every participating organization to translate its source data into this shared vocabulary, OMOP enables seamless data integration and cross-site analytics. Analysts and researchers can write a single query or analysis pipeline that runs unmodified on any OMOP-mapped dataset, regardless of local idiosyncrasies or coding systems. This shared language eliminates ambiguity, fosters reproducibility, and is the foundation for interoperable, multi-center observational research across the international OHDSI community.


In [17]:
# At OMOP's center is a CONCEPT table: every clinical fact is a concept_id.
# Source codes (ICD, SNOMED, LOINC, RxNorm) map to these standard IDs so every
# site speaks the same language.
#
# Columns:
#   concept_id     — the integer that other OMOP tables store as a foreign key
#   concept_name   — human-readable label
#   domain_id      — which OMOP table this concept belongs in
#                    (Gender → person, Condition → condition_occurrence, ...)
#   vocabulary_id  — which coding system it came from
#
# These IDs are ILLUSTRATIVE. Gender 8507/8532 and a few conditions match real
# OHDSI Athena IDs; treat the rest as a teaching stand-in, not the source of truth.
import pandas as pd
concept = pd.DataFrame([
    (8507,    "MALE",                     "Gender",      "Gender"),
    (8532,    "FEMALE",                   "Gender",      "Gender"),
    (320128,  "Essential hypertension",   "Condition",   "SNOMED"),
    (201826,  "Type 2 diabetes mellitus", "Condition",   "SNOMED"),
    (317009,  "Asthma",                   "Condition",   "SNOMED"),
    (3004410, "Hemoglobin A1c",           "Measurement", "LOINC"),
    (3004249, "Systolic blood pressure",  "Measurement", "LOINC"),
    (3025315, "Body weight",              "Measurement", "LOINC"),
    (1308216, "Lisinopril",               "Drug",        "RxNorm"),
    (1503297, "Metformin",                "Drug",        "RxNorm"),
], columns=["concept_id", "concept_name", "domain_id", "vocabulary_id"])
concept


,concept_id,concept_name,domain_id,vocabulary_id
0,8507,MALE,Gender,Gender
1,8532,FEMALE,Gender,Gender
2,320128,Essential hypertension,Condition,SNOMED
3,201826,Type 2 diabetes mellitus,Condition,SNOMED
4,317009,Asthma,Condition,SNOMED
5,3004410,Hemoglobin A1c,Measurement,LOINC
6,3004249,Systolic blood pressure,Measurement,LOINC
7,3025315,Body weight,Measurement,LOINC
8,1308216,Lisinopril,Drug,RxNorm
9,1503297,Metformin,Drug,RxNorm


## 7.3 Building the OMOP tables

At this stage, we translate our original source tables into the standardized OMOP tables. 
String-based IDs from the source data (like patient or visit codes) are converted into 
OMOP-compliant integer foreign keys—such as `person_id` and `visit_occurrence_id`. 
This harmonization enables reliable joins between tables and consistency across datasets.


### Milestone 1 - PERSON

PERSON: This OMOP table standardizes patient identities for all subsequent data integration. 
Each patient from the source data is assigned a unique integer `person_id`, replacing any non-standard string IDs.
The table also stores their gender as a standard concept ID (from the CONCEPT table), year of birth, 
and the original source identifier for traceability.

In [18]:
# PERSON is OMOP's patient table. Source string IDs (P1000, P1001, ...) become
# integer person_id values. Everything else in the CDM will join on person_id.
patients = ehr["patients"].copy()
patients["person_id"] = range(1, len(patients) + 1)  # 1, 2, 3, ... n

# id_map lets later tables translate "P1000" → 1 without looking up patients again.
id_map = dict(zip(patients["patient_id"], patients["person_id"]))

# sex strings → standard gender concept_ids from the CONCEPT table above
gender_concept = {"male": 8507, "female": 8532}

person = pd.DataFrame({
    "person_id": patients["person_id"],
    "gender_concept_id": patients["sex"].map(gender_concept),  # 8507 or 8532
    "year_of_birth": patients["birth_year"],
    # person_source_value keeps the original ID so you can audit the mapping
    "person_source_value": patients["patient_id"],
})
print("PERSON table:")
person.head()


PERSON table:


,person_id,gender_concept_id,year_of_birth,person_source_value
0,1,8532,1937,P1000
1,2,8507,1953,P1001
2,3,8532,1985,P1002
3,4,8532,1939,P1003
4,5,8507,1973,P1004


### Milestone 2 - VISIT_OCCURRENCE, CONDITION_OCCURRENCE, and MEASUREMENT


VISIT_OCCURRENCE: This OMOP table captures information about each patient encounter or visit. 
Each encounter from the source data is assigned a unique integer visit_occurrence_id. 
The type of visit is mapped to a standard visit_concept_id (e.g., inpatient, outpatient, emergency) 
for interoperability. Each row links the visit back to the patient (person_id) and records the start date.

CONDITION_OCCURRENCE: This OMOP table standardizes diagnoses for patients based on their encounters. 
Each condition from the source data is mapped to a standard OMOP concept_id, whenever possible. 
Only diagnoses with a recognized standard concept are retained, and the original label is kept for 
traceability. Each row links the condition to the patient (person_id).

MEASUREMENT: This OMOP table holds quantitative labs and vitals (A1c, blood pressure, weight).
Source observation labels are mapped to standard concept IDs. Measures that are not in our tiny
vocabulary (heart rate, height) are dropped — the same unmapped-rate signal as for conditions.

In [19]:
# VISIT_OCCURRENCE is OMOP's encounter table.
# visit_concept_id uses standard OMOP visit concepts:
#   9201 inpatient, 9202 outpatient/ambulatory/wellness, 9203 emergency
visit_concept = {
    "ambulatory": 9202,
    "emergency": 9203,
    "inpatient": 9201,
    "wellness": 9202,
}
enc = ehr["encounters"].copy()
enc["visit_occurrence_id"] = range(1, len(enc) + 1)
enc["person_id"] = enc["patient_id"].map(id_map)  # P1000 → 1, via the map from PERSON

visit_occurrence = pd.DataFrame({
    "visit_occurrence_id": enc["visit_occurrence_id"],
    "person_id": enc["person_id"],
    "visit_concept_id": enc["encounter_type"].map(visit_concept),
    "visit_start_date": pd.to_datetime(enc["date"]),
})

# CONDITION_OCCURRENCE: source diagnosis labels → standard concept_ids.
# Only three conditions are in our tiny vocabulary; the rest will map to NaN
# and drop out. That drop is itself a data-quality signal (unmapped rate).
cond_to_concept = {
    "Essential hypertension": 320128,
    "Type 2 diabetes mellitus": 201826,
    "Asthma": 317009,
}
cond = ehr["conditions"].copy()
cond["person_id"] = cond["patient_id"].map(id_map)
cond["condition_concept_id"] = cond["condition"].map(cond_to_concept)  # unmapped → NaN

condition_occurrence = (
    cond.dropna(subset=["condition_concept_id"])  # keep only mapped diagnoses
    .assign(condition_concept_id=lambda d: d["condition_concept_id"].astype(int))
    [["person_id", "condition_concept_id", "condition"]]
    .rename(columns={"condition": "condition_source_value"})  # provenance
)

# MEASUREMENT holds quantitative labs and vitals (A1c, blood pressure, weight).
# Heart rate and height are in the source observations but not in our tiny
# concept table, so they are filtered out — same unmapped pattern as conditions.
meas_to_concept = {
    "Hemoglobin A1c": 3004410,
    "Systolic blood pressure": 3004249,
    "Body weight": 3025315,
}
obs = ehr["observations"].copy()
obs["person_id"] = obs["patient_id"].map(id_map)

measurement = (
    obs[obs["observation"].isin(meas_to_concept)]  # keep only mapped measures
    .assign(measurement_concept_id=lambda d: d["observation"].map(meas_to_concept))
    [["person_id", "measurement_concept_id", "value", "unit", "date", "observation"]]
    .rename(columns={
        "value": "value_as_number",          # OMOP's numeric-value column
        "date": "measurement_date",
        "observation": "measurement_source_value",  # provenance: original label
    })
)

print("VISIT_OCCURRENCE:", visit_occurrence.shape,
      "| CONDITION_OCCURRENCE:", condition_occurrence.shape,
      "| MEASUREMENT:", measurement.shape)
condition_occurrence.head()


VISIT_OCCURRENCE: (511, 4) | CONDITION_OCCURRENCE: (109, 3) | MEASUREMENT: (1086, 6)


,person_id,condition_concept_id,condition_source_value
4,3,317009,Asthma
5,3,201826,Type 2 diabetes mellitus
6,4,320128,Essential hypertension
13,9,201826,Type 2 diabetes mellitus
14,10,320128,Essential hypertension


In [20]:
# MEASUREMENT was built in the previous cell with visit_occurrence and
# condition_occurrence. Inspect a few rows: each lab/vital is stored as a
# measurement_concept_id, not as a free-text label.
print("MEASUREMENT table:", measurement.shape)
measurement.head()


MEASUREMENT table: (1086, 6)


,person_id,measurement_concept_id,value_as_number,unit,measurement_date,measurement_source_value
1,1,3025315,55.8,kg,2025-03-17,Body weight
4,1,3004249,127.9,mmHg,2023-05-21,Systolic blood pressure
6,1,3004410,9.5,%,2023-05-21,Hemoglobin A1c
7,1,3025315,92.3,kg,2025-10-05,Body weight
8,1,3004249,158.7,mmHg,2025-10-05,Systolic blood pressure


## 7.4 A standardized analytic query

Now that our data has been transformed into the OMOP Common Data Model (CDM), we can take advantage of its standardized structure to perform analyses that will work at any OMOP-compliant site, without needing to adjust the code for differences in local data tables or variable naming conventions.

For example, if we want to calculate the prevalence of each medical condition represented in our dataset, we can write a query using OMOP's standard fields and concepts. Specifically, we are interested in the prevalence of each mapped condition (e.g., asthma, hypertension, diabetes). The query groups patients by their standardized OMOP concept ID for each condition, counts the number of unique persons with each condition, and then joins this with the OMOP "concept" table. The concept table provides the human-readable names corresponding to each standardized concept ID.

By using standardized concept IDs rather than site-specific strings (such as "Essential hypertension" or "asthma"), our query is robust to differences in how conditions might be labeled or coded at different institutions. This ensures consistent, reproducible, and portable analyses: the code will work "as is" at any site using OMOP, and will always report well-defined concepts with their proper clinical names, regardless of the specific way the local site originally recorded them.


In [21]:
# Because the data is now in OMOP, this query would run UNCHANGED at any OMOP site.
# We count distinct persons per condition_concept_id, then join CONCEPT to get
# human-readable names. The analysis never mentions "Essential hypertension" as
# a string — only the integer 320128 — so site-specific labels cannot break it.
prevalence = (
    condition_occurrence
    .groupby("condition_concept_id")["person_id"].nunique()  # unique patients, not rows
    .rename("n_persons")
    .reset_index()
    .merge(
        concept[["concept_id", "concept_name"]],
        left_on="condition_concept_id",
        right_on="concept_id",
    )
)
prevalence["pct"] = (100 * prevalence["n_persons"] / len(person)).round(1)

print("Standardized prevalence query (portable across OMOP sites):")
print(prevalence[["concept_name", "n_persons", "pct"]].to_string(index=False))


Standardized prevalence query (portable across OMOP sites):
            concept_name  n_persons  pct
Type 2 diabetes mellitus         40 20.0
                  Asthma         39 19.5
  Essential hypertension         30 15.0


In [22]:
# Same idea for a lab: look up A1c by its concept_id, not by the string "Hemoglobin A1c".
# If another site stored the label as "HbA1c" or "glycohemoglobin", this still works.
a1c_id = 3004410
mean_a1c = measurement.loc[
    measurement["measurement_concept_id"] == a1c_id,
    "value_as_number",
].mean()
name = concept.loc[concept["concept_id"] == a1c_id, "concept_name"].iloc[0]
print(f"Mean {name} across cohort: {mean_a1c:.2f} %")


Mean Hemoglobin A1c across cohort: 7.10 %


## 7.5 Other common data models

OMOP is dominant for observational research, but you'll meet others:

| CDM | Origin | Typical use |
|---|---|---|
| **OMOP** | OHDSI | Observational research networks, large-scale analytics |
| **i2b2** | Partners/Harvard | Cohort discovery, local data warehouses |
| **PCORnet** | PCORI | Patient-centered outcomes research |
| **Sentinel** | FDA | Drug/device safety surveillance |

They share the same core idea: **map once, analyze everywhere**.



### 1. OMOP (Observational Medical Outcomes Partnership)
- **Origin:** Developed by OHDSI (Observational Health Data Sciences and Informatics)
- **Typical Use:** Large-scale observational research and analytics; multi-site studies.
- **Core Idea:** Uses a fixed schema (person, visit_occurrence, condition_occurrence, measurement, drug_exposure, etc.) and common vocabulary (concept_id).
- **Python Example:** Query prevalence of a condition standardized by concept_id.


In [23]:
# Find prevalence of 'Asthma' (concept_id = 317009) in a cohort
asthma_id = 317009
asthma_patients = condition_occurrence.loc[
    condition_occurrence["condition_concept_id"] == asthma_id, "person_id"
].nunique()
total_patients = person["person_id"].nunique()
print(f"Asthma prevalence: {100*asthma_patients/total_patients:.1f}%")

Asthma prevalence: 19.5%



### 2. i2b2 (Informatics for Integrating Biology and the Bedside)
- **Origin:** Partners HealthCare / Harvard
- **Typical Use:** Cohort discovery, local clinical data warehouse queries, self-service exploration.
- **Core Idea:** Uses a flexible, hierarchical "star schema" model (fact table + ontology), allowing for fast, ad-hoc queries.
- **Python Example:** Pull encounter counts of patients by sub-tree (e.g., diagnoses starting with 'E11' for diabetes).



In [24]:
# Example stub: i2b2 query for all 'E11%' ICD-10 codes (Type 2 Diabetes)
import pandas as pd
diagnoses = pd.DataFrame({'code': ['E11.9', 'I10'], 'person_id': [1, 2]})
diabetes_patients = diagnoses[diagnoses['code'].str.startswith('E11')]['person_id'].nunique()
print(f"Type 2 diabetes patients (i2b2): {diabetes_patients}")

Type 2 diabetes patients (i2b2): 1



### 3. PCORnet (Patient-Centered Outcomes Research Network)
- **Origin:** PCORI (Patient-Centered Outcomes Research Institute)
- **Typical Use:** National-scale, multi-system patient-centered outcomes research.
- **Core Idea:** Wide tables, common vocabulary mappings, supports pragmatic clinical trial and observational studies.




In [25]:
# Example stub: PCORnet procedure table analysis
procedures = pd.DataFrame({'px_code': ['93000', '80050'], 'person_id': [3, 4]})
ecg_count = procedures[procedures['px_code'] == '93000']['person_id'].nunique()
print(f"Patients with ECG procedure (PCORnet): {ecg_count}")

Patients with ECG procedure (PCORnet): 1



### 4. Sentinel
- **Origin:** FDA
- **Typical Use:** Drug and device safety surveillance, regulatory queries.
- **Core Idea:** Designed for regulatory queries at scale, federated querying across data partners, focused tables (dispensing, enrollment, etc.).




In [26]:
dispensing = pd.DataFrame({'ndc': ['00093754456', '00591088810'], 'person_id': [5, 6]})
drug_count = dispensing[dispensing['ndc'] == '00093754456']['person_id'].nunique()
print(f"Patients dispensed statins (Sentinel): {drug_count}")

Patients dispensed statins (Sentinel): 1


## Exercises

1. Add **Asthma** patients to the prevalence query and confirm the concept join
   still resolves names correctly.
2. Build a `drug_exposure` table by mapping the medications to the two drug
   concepts in the vocabulary, and count exposures per drug.
3. Report how many condition rows were **unmapped** (fell out because the source
   label had no standard concept) — a key metric of mapping completeness.



## Key takeaways

- A CDM turns **N bespoke extractions into one mapping + one analysis**.
- OMOP's core is the **standardized vocabulary** (`concept_id`) plus a fixed schema
  (`person`, `visit_occurrence`, `condition_occurrence`, `measurement`, `drug_exposure`).
- **Provenance** (`*_source_value`) is preserved so mappings stay auditable.



---
*Next: Module 8 - Data Quality and Validation.*
